# Backtest Swing Trading: Repsol y ArcelorMittal  Esta es una prueba de cambio github  OOOOOOOOOOOOO
Estrategia: **Breakout (Donchian Channel) con filtro de volumen + trailing stop por ATR**

Este notebook:
1. Descarga 3 años de datos diarios de Repsol (`REP.MC`) y ArcelorMittal (`MT.AS`)
2. Aplica una estrategia de ruptura de rango (breakout) con confirmación de volumen
3. Gestiona el riesgo con un trailing stop basado en ATR
4. Muestra estadísticas de rendimiento y una gráfica del equity curve

⚠️ **Esto es material educativo, no es asesoramiento financiero.** Los resultados pasados no garantizan resultados futuros. Antes de operar con dinero real, valida cualquier estrategia con más datos, costes reales de comisión/slippage y gestión de riesgo adecuada.

## 1. Instalar librerías

In [ ]:
!pip install yfinance backtesting --quiet

## 2. Importar librerías

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## 3. Descargar datos (últimos 3 años)

- `REP.MC` = Repsol (Bolsa de Madrid)
- `MT.AS` = ArcelorMittal (Euronext Ámsterdam, su listado principal y más líquido). 
  También cotiza en Madrid como `MTS.MC` si prefieres esa referencia.

In [ ]:
tickers = {
    "Repsol": "REP.MC",
    "ArcelorMittal": "MT.AS",
}

data = {}
for name, ticker in tickers.items():
    df = yf.download(ticker, period="3y", interval="1d", auto_adjust=True, progress=False)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]  # aplanar columnas si vienen en MultiIndex
    df = df.dropna()
    data[name] = df
    print(f"{name} ({ticker}): {len(df)} velas diarias, desde {df.index[0].date()} hasta {df.index[-1].date()}")

## 4. Definir la estrategia

**Lógica:**
- Canal Donchian de 20 días (máximo/mínimo de los últimos 20 días)
- Entrada LARGA cuando el precio rompe el máximo de 20 días **y** el volumen supera su media de 20 días (confirma la ruptura)
- Entrada CORTA cuando rompe el mínimo de 20 días con el mismo filtro de volumen
- Stop-loss inicial y trailing stop basados en ATR (2x ATR de 14 días)
- Tamaño de posición: fracción fija de la cartera (ajustable)

In [ ]:
def ATR(high, low, close, n=14):
    high, low, close = pd.Series(high), pd.Series(low), pd.Series(close)
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(n).mean()


class DonchianBreakout(Strategy):
    n_channel = 20      # ventana del canal de ruptura
    n_atr = 14           # ventana del ATR
    atr_mult = 2.0        # multiplicador del ATR para el stop
    vol_mult = 1.0        # el volumen debe superar (vol_mult x media de volumen) para confirmar

    def init(self):
        close = pd.Series(self.data.Close)
        high = pd.Series(self.data.High)
        low = pd.Series(self.data.Low)
        volume = pd.Series(self.data.Volume)

        self.upper = self.I(lambda x: pd.Series(x).rolling(self.n_channel).max(), self.data.High)
        self.lower = self.I(lambda x: pd.Series(x).rolling(self.n_channel).min(), self.data.Low)
        self.vol_avg = self.I(lambda x: pd.Series(x).rolling(self.n_channel).mean(), self.data.Volume)
        self.atr = self.I(ATR, self.data.High, self.data.Low, self.data.Close, self.n_atr)

    def next(self):
        price = self.data.Close[-1]
        vol_ok = self.data.Volume[-1] > self.vol_mult * self.vol_avg[-1]
        atr_val = self.atr[-1]

        if np.isnan(atr_val) or np.isnan(self.upper[-1]) or np.isnan(self.lower[-1]):
            return

        # Gestionar trailing stop si hay posición abierta
        if self.position:
            if self.position.is_long:
                new_stop = price - self.atr_mult * atr_val
                for trade in self.trades:
                    trade.sl = max(trade.sl or -np.inf, new_stop)
            elif self.position.is_short:
                new_stop = price + self.atr_mult * atr_val
                for trade in self.trades:
                    trade.sl = min(trade.sl or np.inf, new_stop)

        # Señales de entrada (solo si no hay posición)
        if not self.position:
            if price >= self.upper[-1] and vol_ok:
                sl = price - self.atr_mult * atr_val
                self.buy(sl=sl)
            elif price <= self.lower[-1] and vol_ok:
                sl = price + self.atr_mult * atr_val
                self.sell(sl=sl)

## 5. Ejecutar el backtest para cada valor

Parámetros de la cuenta simulada:
- Capital inicial: 10.000 €
- Comisión: 0.1% por operación (ajusta según tu bróker)
- Sin apalancamiento (`margin=1`)

In [ ]:
results = {}

for name, df in data.items():
    bt = Backtest(df, DonchianBreakout, cash=10_000, commission=0.001, margin=1, exclusive_orders=True)
    stats = bt.run()
    results[name] = (bt, stats)
    print(f"\n{'='*20} {name} {'='*20}")
    print(stats)

## 6. Comparativa rápida de métricas clave

In [ ]:
summary = pd.DataFrame({
    name: {
        "Retorno total [%]": stats["Return [%]"],
        "Retorno anualizado [%]": stats["Return (Ann.) [%]"],
        "Buy & Hold [%]": stats["Buy & Hold Return [%]"],
        "Máx. drawdown [%]": stats["Max. Drawdown [%]"],
        "Ratio de aciertos [%]": stats["Win Rate [%]"],
        "Nº operaciones": stats["# Trades"],
        "Sharpe": stats["Sharpe Ratio"],
    }
    for name, (bt, stats) in results.items()
}).T

summary.round(2)

## 7. Gráfica del equity curve (una por valor)

Ejecuta esta celda una vez por valor, cambiando `nombre_valor`.

In [ ]:
nombre_valor = "Repsol"   # cambia a "ArcelorMittal" para ver el otro

bt, stats = results[nombre_valor]
bt.plot()

## 8. Cómo seguir explorando

Prueba a ajustar estos parámetros y vuelve a correr las celdas 5-7:

- `n_channel`: ventana del canal de ruptura (prueba 10, 20, 55 días)
- `atr_mult`: qué tan ajustado o amplio es el stop (1.5x - 3x)
- `vol_mult`: exige más o menos confirmación de volumen

También puedes:
- Cambiar el `period="3y"` por `"5y"` o fechas concretas con `start=` y `end=`
- Añadir un filtro de tendencia (ej. solo operar largo si el precio está sobre su media de 200 sesiones)
- Optimizar parámetros automáticamente con `bt.optimize(...)` (cuidado con el sobreajuste / overfitting)

**Recuerda:** este es un ejercicio educativo. Antes de usar cualquier estrategia con dinero real, valida con datos out-of-sample, incluye costes realistas (comisiones, slippage) y define bien tu gestión de riesgo por operación.